# Data Loading

Verify the two original UCI source files, inspect the encounter schema, and summarize patient coverage and original target labels. The mapping file is a reference for interpreting source codes.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_loading import (
    load_ids_mapping,
    load_raw_encounters,
    profile_raw_columns,
    summarize_raw_encounters,
)

## Source Integrity

The loaders verify the exact SHA-256 digests, column order, and row counts. Both CSV files are read as text so that identifiers, empty fields, and literal markers retain their original representation.

In [ ]:
encounters = load_raw_encounters()
mapping = load_ids_mapping()

file_table = [
    "| Source file | Data rows | Columns | Bytes | SHA-256 |",
    "| --- | ---: | ---: | ---: | --- |",
]
for table in (encounters, mapping):
    file_table.append(
        f"| `{table.path.name}` | {len(table.rows):,} | {len(table.columns)} | "
        f"{table.size_bytes:,} | `{table.sha256}` |"
    )
display(Markdown("\n".join(file_table)))

## Encounter Coverage and Original Labels

Each row has a unique encounter ID. Patient IDs can repeat across encounters. The source labels retain their three original categories.

In [ ]:
summary = summarize_raw_encounters(encounters)
print(f"Encounters: {summary['encounters']:,}")
print(f"Unique patients: {summary['patients']:,}")
print(f"Patients with multiple encounters: {summary['patients_with_multiple_encounters']:,}")

label_table = ["| Original label | Encounters | Share |", "| --- | ---: | ---: |"]
for label, count in summary["target_counts"].items():
    label_table.append(f"| `{label}` | {count:,} | {count / summary['encounters']:.2%} |")
display(Markdown("\n".join(label_table)))

## Encounter Schema and Literal Markers

All loaded values are strings. Logical types describe the source fields, including integer counts and categorical codes. The marker columns count distinct source representations. The string `None` in the laboratory fields is preserved as a recorded category.

In [ ]:
profile = profile_raw_columns(encounters)
column_table = [
    "| Column | Logical type | Empty | `?` | `None` | `Unknown/Invalid` |",
    "| --- | --- | ---: | ---: | ---: | ---: |",
]
for row in profile:
    column_table.append(
        f"| `{row['column']}` | {row['logical_type']} | "
        f"{row['']:,} | {row['?']:,} | {row['None']:,} | {row['Unknown/Invalid']:,} |"
    )
display(Markdown("\n".join(column_table)))